<a href="https://colab.research.google.com/github/GabrielDarG/Dataset_Alugueis_Bicicleta/blob/main/TheTrueDataScience.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📈 Projeto de Previsão de Aluguel de Bicicletas (Seoul)

Objetivo: Construir um modelo de Machine Learning capaz de prever a demanda (quantidade de aluguéis) de bicicletas com base em dados como clima, data e hora.

## **Importação das Bibliotecas e Carregamento dos Dados**

Primeiro, vamos importar todas as ferramentas (bibliotecas) que precisaremos para nosso projeto, e vamos carregar nosso conjunto de dados

In [ ]:
import pandas as pd
import numpy as np
import joblib
import itertools
import time

import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, mean_squared_error, r2_score

In [ ]:
#carrega o banco de dados no df
url = "https://raw.githubusercontent.com/GabrielDarG/Dataset_Alugueis_Bicicleta/main/SeoulBikeData.csv"
df = pd.read_csv(url, encoding="latin1")

## **Passo 1: Limpeza e Engenharia de Features**

Nesta etapa crucial, limpamos os dados removendo colunas irrelevantes e traduzindo os nomes. Em seguida, aplicamos engenharia de features: extraímos informações da data, transformamos horas e meses em features cíclicas com seno/cosseno para que o modelo entenda a continuidade (ex: 23h perto de 0h), e convertemos texto como 'Primavera' em colunas numéricas usando One-Hot Encoding. Ao final, o dataset ficou totalmente numérico e otimizado para o treinamento do modelo.

In [ ]:
# Remover coluna desnecessária
df.drop(columns=["Dew point temperature(°C)", "Visibility (10m)", "Snowfall (cm)", "Solar Radiation (MJ/m2)"], inplace=True)

# Renomear colunas para português
df.rename(columns={
    "Rented Bike Count": "Qtd_Aluguel",
    "Hour": "Hora",
    "Temperature(°C)": "Temp_C",
    "Humidity(%)": "Umidade",
    "Wind speed (m/s)": "Vento",
    "Rainfall(mm)": "Chuva_mm",
    "Seasons": "Estacao",
    "Holiday": "Feriado",
    "Functioning Day": "Dia_Funcionamento"
}, inplace=True)

# Converte o campo "date" para ser um tipo date
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Extrair o mês
df['Mes'] = df['Date'].dt.month

# Extrair o dia da semana
df["Dia_Semana"] = df["Date"].dt.day_name()

# Extrair o dia
df['Dia'] = df['Date'].dt.day

df.drop(columns=['Date'], inplace=True)


dias_semana_map = {
    "Monday": "Segunda", "Tuesday": "Terça", "Wednesday": "Quarta",
    "Thursday": "Quinta", "Friday": "Sexta",
    "Saturday": "Sábado", "Sunday": "Domingo"
}

df["Dia_Semana"] = df["Dia_Semana"].map(dias_semana_map)

estacao_map = {
    "Winter": "Inverno", "Spring": "Primavera", "Summer": "Verão", "Autumn": "Outono"
}
df["Estacao"] = df["Estacao"].map(estacao_map)


# --- INÍCIO DA ENGENHARIA DE FEATURES AVANÇADA ---

# 1. Tratar a ciclicidade da HORA
# Isso cria um "círculo" de 24 horas para o modelo entender a proximidade de 23h e 0h.
df['hora_sin'] = np.sin(2 * np.pi * df['Hora']/24.0)
df['hora_cos'] = np.cos(2 * np.pi * df['Hora']/24.0)

# 2. Tratar a ciclicidade do MÊS
# Ajuda o modelo a entender que Dezembro está perto de Janeiro.
df['mes_sin'] = np.sin(2 * np.pi * df['Mes']/12.0)
df['mes_cos'] = np.cos(2 * np.pi * df['Mes']/12.0)

# Vamos remover as colunas originais que não precisamos mais
df.drop(columns=['Hora', 'Mes'], inplace=True)


# --- FIM DA ENGENHARIA DE FEATURES AVANÇADA ---



# Use o get_dummies do pandas para o One-Hot Encoding
df = pd.get_dummies(df, columns=['Estacao', 'Feriado', "Dia_Funcionamento",'Dia_Semana'], drop_first=True)
# o drop_first = evita o multicolinearidade, removendo uma das colunas de Seasons, deixando apenas 3
print(df.head())

   Qtd_Aluguel  Temp_C  Umidade  Vento  Chuva_mm  Dia  hora_sin  hora_cos  \
0          254    -5.2       37    2.2       0.0    1  0.000000  1.000000   
1          204    -5.5       38    0.8       0.0    1  0.258819  0.965926   
2          173    -6.0       39    1.0       0.0    1  0.500000  0.866025   
3          107    -6.2       40    0.9       0.0    1  0.707107  0.707107   
4           78    -6.0       36    2.3       0.0    1  0.866025  0.500000   

        mes_sin  mes_cos  ...  Estacao_Primavera  Estacao_Verão  \
0 -2.449294e-16      1.0  ...              False          False   
1 -2.449294e-16      1.0  ...              False          False   
2 -2.449294e-16      1.0  ...              False          False   
3 -2.449294e-16      1.0  ...              False          False   
4 -2.449294e-16      1.0  ...              False          False   

   Feriado_No Holiday  Dia_Funcionamento_Yes  Dia_Semana_Quarta  \
0                True                   True              False   


## **Passo 2: Preparação para o Treinamento**

### X: As features (dados de entrada que o modelo usa para prever).

### y: O alvo (o que queremos prever, ou seja, Qtd_Aluguel).

In [ ]:
# Separando dados em variável preditoras (X) da variável alvo (y)
X = df.drop('Qtd_Aluguel', axis=1) # Todas as colunas, exceto a de aluguéis
y = df['Qtd_Aluguel'] # A coluna de aluguéis

# Salvar as colunas do modelo para usar na API depois
colunas_do_modelo = list(X.columns)
joblib.dump(colunas_do_modelo, "colunas_do_modelo.pkl")
print(f"Colunas do modelo salvas em 'colunas_do_modelo.pkl'")

# Dividir os dados em conjuntos de treino e teste
# test_size=0.20 significa que 20% dos dados serão para o teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=40)

Colunas do modelo salvas em 'colunas_do_modelo.pkl'


## **Passo 3: Competição de Modelos**

### Vamos treinar 4 modelos populares com suas configurações padrão para ver qual se sai melhor inicialmente.

In [ ]:
print("--- Iniciando Treinamento Baseline ---")

# --- Modelo 1: XGBoost ---
print("\nTreinando XGBoost...")
xgb_model = xgb.XGBRegressor(random_state=40)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_r2 = r2_score(y_test, xgb_pred)
print("Resultado do XGBoost (Baseline)")
print(f"Erro Quadrático Médio (MSE): {xgb_mse:.2f}")
print(f"R-quadrado (R² Score): {xgb_r2:.2f}")

# --- Modelo 2: LightGBM ---
print("\nTreinando LightGBM...")
lgbm_model = LGBMRegressor(random_state=40, verbose=-1)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)
lgbm_mse = mean_squared_error(y_test, lgbm_pred)
lgbm_r2 = r2_score(y_test, lgbm_pred)
print("Resultado do LightGBM (Baseline)")
print(f"Erro Quadrático Médio (MSE): {lgbm_mse:.2f}")
print(f"R-quadrado (R² Score): {lgbm_r2:.2f}")

# --- Modelo 3: Gradient Boosting (Scikit-learn) ---
print("\nTreinando Gradient Boosting...")
gb_model = GradientBoostingRegressor(random_state=40)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_mse = mean_squared_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)
print("Resultado do Gradient Boosting (Baseline)")
print(f"Erro Quadrático Médio (MSE): {gb_mse:.2f}")
print(f"R-quadrado (R² Score): {gb_r2:.2f}")

# --- Modelo 4: Random Forest ---
print("\nTreinando Random Forest...")
rf_model = RandomForestRegressor(random_state=40)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)
print("\nResultado do Random Forest (Baseline)")
print(f"Erro Quadrático Médio (MSE): {rf_mse:.2f}")
print(f"R-quadrado (R² Score): {rf_r2:.2f}")

--- Iniciando Treinamento Baseline ---

Treinando XGBoost...
Resultado do XGBoost (Baseline)
Erro Quadrático Médio (MSE): 31039.71
R-quadrado (R² Score): 0.93

Treinando LightGBM...
Resultado do LightGBM (Baseline)
Erro Quadrático Médio (MSE): 30704.53
R-quadrado (R² Score): 0.93

Treinando Gradient Boosting...
Resultado do Gradient Boosting (Baseline)
Erro Quadrático Médio (MSE): 65227.07
R-quadrado (R² Score): 0.85

Treinando Random Forest...

Resultado do Random Forest (Baseline)
Erro Quadrático Médio (MSE): 43660.99
R-quadrado (R² Score): 0.90


In [ ]:
joblib.dump(xgb_model, "modelo_previsao_bicicletas.pkl")
joblib.dump(lgbm_model, "modelo_previsao_bicicletas_LGBM.pkl")
joblib.dump(rf_model, "modelo_previsao_bicicletas_RandomForest.pkl")
joblib.dump(gb_model, "modelo_previsao_bicicletas_GradientBoosting.pkl")

['modelo_previsao_bicicletas_GradientBoosting.pkl']

### Testando na pratica os 4 modelos

In [ ]:
try:
    modelo_carregado = joblib.load("modelo_previsao_bicicletas_LGBM.pkl") #Altere entre os modelos escolhidos la em cima, os .PKL
    colunas_do_modelo = joblib.load("colunas_do_modelo.pkl")
    print("✅ Modelo e colunas carregados com sucesso!")
except FileNotFoundError:
    print("🛑 ERRO: Arquivos 'modelo_lgbm_refinado.pkl' ou 'colunas_do_modelo.pkl' não encontrados.")
    print("Verifique se os nomes dos arquivos estão corretos e no mesmo diretório.")
    # Se der erro aqui, pare o script.
    exit()


# Altere estes valores para fazer diferentes testes!
novos_dados_para_prever = {
    'Hora': 18,                       # Hora do dia (0 a 23)
    'Temp_C': 25.0,                   # Temperatura em Celsius
    'Umidade': 60,                    # Umidade em %
    'Vento': 2.2,                     # Velocidade do vento em m/s
    'Chuva_mm': 0.0,                  # Precipitação em mm
    'Estacao': 'Primavera',           # 'Inverno', 'Primavera', 'Verão', 'Outono'
    'Feriado': 'No Holiday',          # 'Holiday' ou 'No Holiday'
    'Dia_Funcionamento': 'Yes',       # 'Yes' ou 'No'
    'Mes': 10,                        # Mês do ano (1 a 12)
    'Dia_Semana': 'Quarta',           # 'Segunda', 'Terça', etc.
    'Dia': 15                         # Dia do mês (não usado pelo modelo, mas bom ter)
}

print("\n⚙️  Dados de entrada para a previsão:")
print(novos_dados_para_prever)


# --- PASSO 3: FUNÇÃO PARA PREPARAR OS DADOS ---
# Esta função aplica EXATAMENTE as mesmas transformações dos dados de treino.
def preparar_dados(novos_dados, colunas_modelo):
    # Converte o dicionário em um DataFrame de uma linha
    df = pd.DataFrame([novos_dados])

    # --- Engenharia de Features Cíclicas (copiado do seu notebook) ---
    df['hora_sin'] = np.sin(2 * np.pi * df['Hora']/24.0)
    df['hora_cos'] = np.cos(2 * np.pi * df['Hora']/24.0)
    df['mes_sin'] = np.sin(2 * np.pi * df['Mes']/12.0)
    df['mes_cos'] = np.cos(2 * np.pi * df['Mes']/12.0)
    df.drop(columns=['Hora', 'Mes'], inplace=True)

    # --- One-Hot Encoding (o passo mais importante) ---
    # Aplica o get_dummies nas colunas categóricas
    df = pd.get_dummies(df, columns=['Estacao', 'Feriado', 'Dia_Funcionamento', 'Dia_Semana'])

    # --- Alinhamento de Colunas ---
    # Garante que o DataFrame final tenha exatamente as mesmas colunas (na mesma ordem)
    # que o modelo usou para treinar. Colunas que não existem nos novos dados serão
    # criadas com o valor 0.
    df_final = df.reindex(columns=colunas_modelo, fill_value=0)

    return df_final


# --- PASSO 4: PREPARAR OS DADOS, FAZER A PREVISÃO E EXIBIR ---

# Prepara os dados usando a função
dados_prontos = preparar_dados(novos_dados_para_prever, colunas_do_modelo)

# Faz a previsão
previsao_bruta = modelo_carregado.predict(dados_prontos)

# Garante que a previsão não seja negativa e arredonda para um número inteiro
previsao_final = round(max(0, previsao_bruta[0]))

print("\n-------------------------------------------")
print(f"📈 PREVISÃO DE ALUGUÉIS DE BICICLETA:")
print(f"Para as condições informadas, a demanda estimada é de aproximadamente {previsao_final} bicicletas.")
print("-------------------------------------------")

✅ Modelo e colunas carregados com sucesso!

⚙️  Dados de entrada para a previsão:
{'Hora': 18, 'Temp_C': 25.0, 'Umidade': 60, 'Vento': 2.2, 'Chuva_mm': 0.0, 'Estacao': 'Primavera', 'Feriado': 'No Holiday', 'Dia_Funcionamento': 'Yes', 'Mes': 10, 'Dia_Semana': 'Quarta', 'Dia': 15}

-------------------------------------------
📈 PREVISÃO DE ALUGUÉIS DE BICICLETA:
Para as condições informadas, a demanda estimada é de aproximadamente 2773 bicicletas.
-------------------------------------------


##**Passo 4: Treinando Modelo**

### LGBM

In [ ]:
# 1. Definir o espaço de parâmetros para testar
param_dist = {
    'n_estimators': [200, 400, 600, 800, 1000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [20, 31, 40, 50],
    'max_depth': [-1, 10, 20]
}

# 2. Configurar a busca aleatória
lgbm = LGBMRegressor(random_state=40, verbose=-1)
random_search = RandomizedSearchCV(
    lgbm,
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring='neg_mean_squared_log_error',
    random_state=40,
    n_jobs=-1
)

# 3. Executar a busca (pode demorar alguns minutos)
print("Iniciando a otimização RÁPIDA do LGBM...")
random_search.fit(X_train, y_train)
print("Otimização concluída!")

# 4. Pegar o melhor modelo e avaliar
best_lgbm_random = random_search.best_estimator_
lgbm_pred_otimizado = best_lgbm_random.predict(X_test)
lgbm_pred_otimizado[lgbm_pred_otimizado < 0] = 0
lgbm_rmsle_otimizado = np.sqrt(mean_squared_log_error(y_test, lgbm_pred_otimizado))

print(f"\nMelhores parâmetros (LGBM Rápido): {random_search.best_params_}")
print(f"RMSLE do LGBM (Otimização Rápida): {lgbm_rmsle_otimizado:.4f}")

Iniciando a otimização RÁPIDA do LGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Otimização concluída!

Melhores parâmetros (LGBM Rápido): {'num_leaves': 31, 'n_estimators': 600, 'max_depth': -1, 'learning_rate': 0.05}
RMSLE do LGBM (Otimização Rápida): 0.8270


### XGBoost

In [ ]:
# 1. Definir o espaço de parâmetros do XGBoost
param_dist_xgb = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'n_estimators': [200, 400, 600]
}

# 2. Configurar a busca aleatória
xgb_model = xgb.XGBRegressor(random_state=40, objective='reg:squarederror')
random_search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=15,
    cv=5,
    scoring='neg_mean_squared_log_error',
    random_state=40,
    n_jobs=-1
)

# 3. Executar a busca
print("\nIniciando a otimização RÁPIDA do XGBoost...")
random_search_xgb.fit(X_train, y_train)
print("Otimização concluída!")

# 4. Pegar o melhor modelo e avaliar
best_xgb_random = random_search_xgb.best_estimator_
xgb_pred_otimizado = best_xgb_random.predict(X_test)
xgb_pred_otimizado[xgb_pred_otimizado < 0] = 0
xgb_rmsle_otimizado = np.sqrt(mean_squared_log_error(y_test, xgb_pred_otimizado))

print(f"\nMelhores parâmetros (XGB Rápido): {random_search_xgb.best_params_}")
print(f"RMSLE do XGBoost (Otimização Rápida): {xgb_rmsle_otimizado:.4f}")



Iniciando a otimização RÁPIDA do XGBoost...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Otimização concluída!

Melhores parâmetros (XGB Rápido): {'subsample': 0.9, 'n_estimators': 600, 'max_depth': 7, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
RMSLE do XGBoost (Otimização Rápida): 0.8393


### Comparando os dois modelos

In [ ]:
# Calcular MSE e R² para os modelos otimizados na primeira rodada
lgbm_mse_otimizado = mean_squared_error(y_test, lgbm_pred_otimizado)
lgbm_r2_otimizado = r2_score(y_test, lgbm_pred_otimizado)

xgb_mse_otimizado_rd1 = mean_squared_error(y_test, xgb_pred_otimizado)
xgb_r2_otimizado_rd1 = r2_score(y_test, xgb_pred_otimizado)


print("\n--- Comparação (Round 1 - RandomizedSearch) ---")

print("\nResultados do LightGBM (Otimização Rápida):")
print(f"  RMSLE: {lgbm_rmsle_otimizado:.4f}")
print(f"  MSE:   {lgbm_mse_otimizado:.2f}")
print(f"  R²:    {lgbm_r2_otimizado:.4f}")

print("\nResultados do XGBoost (Otimização Rápida):")
print(f"  RMSLE: {xgb_rmsle_otimizado:.4f}")
print(f"  MSE:   {xgb_mse_otimizado_rd1:.2f}")
print(f"  R²:    {xgb_r2_otimizado_rd1:.4f}")



--- Comparação (Round 1 - RandomizedSearch) ---

Resultados do LightGBM (Otimização Rápida):
  RMSLE: 0.8270
  MSE:   23384.95
  R²:    0.9459

Resultados do XGBoost (Otimização Rápida):
  RMSLE: 0.8393
  MSE:   25023.71
  R²:    0.9421


### Refinando o Melhor modelo (LightGBM)

In [ ]:
print("\nIniciando a busca REFINADA (GridSearchCV) para o LGBM...")

# 1. Definir uma grade pequena e focada nos melhores parâmetros encontrados
param_grid_refined = {
    'learning_rate': [0.03, 0.05, 0.07],
    'n_estimators': [500, 600, 700],
    'num_leaves': [25, 31, 35],
    'max_depth': [-1]
}

# 2. Configurar o GridSearchCV
lgbm = LGBMRegressor(random_state=40, verbose=-1)
grid_search = GridSearchCV(
    lgbm,
    param_grid=param_grid_refined,
    cv=5,
    scoring='neg_mean_squared_log_error',
    n_jobs=-1
)

# 3. Executar a busca (pode demorar alguns minutos)
grid_search.fit(X_train, y_train)
print("Busca refinada concluída!")

# 4. Avaliar o modelo final, ultra-otimizado
ultra_best_lgbm = grid_search.best_estimator_
lgbm_pred_refinado = ultra_best_lgbm.predict(X_test)
lgbm_pred_refinado[lgbm_pred_refinado < 0] = 0
rmsle_refinado = np.sqrt(mean_squared_log_error(y_test, lgbm_pred_refinado))

print(f"\nMelhores parâmetros da busca refinada: {grid_search.best_params_}")
print(f"RMSLE do LGBM (Otimização Refinada): {rmsle_refinado:.4f}")

# Calcular MSE e R² para o modelo refinado
lgbm_mse_refinado_final = mean_squared_error(y_test, lgbm_pred_refinado)
lgbm_r2_refinado_final = r2_score(y_test, lgbm_pred_refinado)

print("\nMétricas adicionais do modelo refinado:")
print(f"  MSE:   {lgbm_mse_refinado_final:.2f}")
print(f"  R²:    {lgbm_r2_refinado_final:.4f}")


Iniciando a busca REFINADA (GridSearchCV) para o LGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Busca refinada concluída!

Melhores parâmetros da busca refinada: {'learning_rate': 0.03, 'max_depth': -1, 'n_estimators': 500, 'num_leaves': 25}
RMSLE do LGBM (Otimização Refinada): 0.8009

Métricas adicionais do modelo refinado:
  MSE:   28572.75
  R²:    0.9339


In [ ]:
# 5. Salvar o modelo campeão
# O modelo 'ultra_best_lgbm' (do grid_search) é o nosso modelo final!
joblib.dump(ultra_best_lgbm, "modelo_lgbm_refinado.pkl")
print("\n✅ Modelo campeão salvo como 'modelo_lgbm_refinado.pkl'")


✅ Modelo campeão salvo como 'modelo_lgbm_refinado.pkl'


## Passo 5: Simulação (Usando o Modelo na Prática)

Agora que temos nosso modelo campeão salvo, esta célula simula o fluxo completo de uma previsão real: carrega o modelo, recebe novos dados (como se viessem de um site), prepara esses dados com uma função e, finalmente, exibe a previsão.

In [ ]:
# --- 1. Carregar o Modelo e as Colunas ---
try:
    modelo_carregado = joblib.load("modelo_lgbm_refinado.pkl")
    colunas_do_modelo = joblib.load("colunas_do_modelo.pkl")
    print("✅ Modelo e colunas carregados com sucesso!")
except FileNotFoundError:
    print("🛑 ERRO: Arquivos .pkl não encontrados. Rode os passos anteriores.")
    raise

# --- 2. Simular Dados de Entrada (do Site) ---
# Altere estes valores para fazer diferentes testes!
novos_dados_para_prever = {
    'Hora': 18,
    'Temp_C': 25.0,
    'Umidade': 60,
    'Vento': 2.2,
    'Chuva_mm': 0.0,
    'Estacao': 'Primavera',
    'Feriado': 'No Holiday',
    'Dia_Funcionamento': 'Yes',
    'Mes': 10,
    'Dia_Semana': 'Quarta',
    'Dia': 15
}
print("\n⚙️  Dados de entrada para a previsão:")
print(novos_dados_para_prever)


# --- 3. Função de Pré-Processamento ---
# Esta é a função (exatamente como a do seu Flask) que prepara os dados do usuário.
def preparar_dados(novos_dados, colunas_modelo):
    df = pd.DataFrame([novos_dados])
    df['hora_sin'] = np.sin(2 * np.pi * df['Hora']/24.0)
    df['hora_cos'] = np.cos(2 * np.pi * df['Hora']/24.0)
    df['mes_sin'] = np.sin(2 * np.pi * df['Mes']/12.0)
    df['mes_cos'] = np.cos(2 * np.pi * df['Mes']/12.0)
    df.drop(columns=['Hora', 'Mes'], inplace=True)
    df = pd.get_dummies(df, columns=['Estacao', 'Feriado', 'Dia_Funcionamento', 'Dia_Semana'])
    df_final = df.reindex(columns=colunas_modelo, fill_value=0)
    return df_final


# --- 4. Fazer a Previsão Final! ---
# Prepara os dados usando a função
dados_prontos = preparar_dados(novos_dados_para_prever, colunas_do_modelo)

# Faz a previsão
previsao_bruta = modelo_carregado.predict(dados_prontos)

# Garante que a previsão não seja negativa e arredonda
previsao_final = round(max(0, previsao_bruta[0]))

print("\n-------------------------------------------")
print(f"📈 PREVISÃO DE ALUGUÉIS DE BICICLETA:")
print(f"Para as condições informadas, a demanda estimada é de aproximadamente {previsao_final} bicicletas.")
print("-------------------------------------------")


✅ Modelo e colunas carregados com sucesso!

⚙️  Dados de entrada para a previsão:
{'Hora': 18, 'Temp_C': 25.0, 'Umidade': 60, 'Vento': 2.2, 'Chuva_mm': 0.0, 'Estacao': 'Primavera', 'Feriado': 'No Holiday', 'Dia_Funcionamento': 'Yes', 'Mes': 10, 'Dia_Semana': 'Quarta', 'Dia': 15}

-------------------------------------------
📈 PREVISÃO DE ALUGUÉIS DE BICICLETA:
Para as condições informadas, a demanda estimada é de aproximadamente 2689 bicicletas.
-------------------------------------------
